# Dataset Merge Walkthrough

This generated notebook is the recipe companion for
`examples/api_dataset_merge_demo.py`.

Combine raw and processed datasets, handle mnemonic collisions safely, and inspect the merge history captured in dataset provenance.

What you will practice in this walkthrough:

- Separate raw acquisition data from derived or QC datasets before merging them.
- Choose an explicit collision policy instead of silently overwriting matching mnemonics.
- Use merge-history provenance to understand how the final working dataset was assembled.

Prerequisites:

- the repository dependencies installed in the active environment

Release follow-up:

- this notebook currently adds the local `src/` and `examples/` paths so it
  can run from a source checkout
- after publishing, switch the recipe toward installed-package-first imports
  and validate it against the published distribution


In [ ]:
import sys
from pathlib import Path

# Walk upward from the current working directory until we find the
# repository root. This keeps the notebook runnable whether Jupyter was
# launched from the repo root or from examples/notebooks.
cwd = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in (cwd, *cwd.parents)
    if (path / "examples").exists() and (path / "src").exists()
)

EXAMPLES_DIR = REPO_ROOT / "examples"
SRC_DIR = REPO_ROOT / "src"
WORKSPACE_DIR = REPO_ROOT / "workspace"
WORKSPACE_RENDERS = WORKSPACE_DIR / "renders"
WORKSPACE_RENDERS.mkdir(parents=True, exist_ok=True)

for candidate in (SRC_DIR, EXAMPLES_DIR):
    candidate_text = str(candidate)
    if candidate_text not in sys.path:
        sys.path.insert(0, candidate_text)


In [ ]:
# Display the source directly in the notebook so the recipe is easy to
# read and copy from without opening another file.
from IPython.display import Code, display

source_path = EXAMPLES_DIR / "api_dataset_merge_demo.py"
display(Code(source_path.read_text(), language="python"))


In [ ]:
# Import the example module and the merge builder helper used in the script.
import api_dataset_merge_demo as demo
from wellplot import DatasetBuilder

# Build the raw and processed inputs separately so you can inspect what is
# about to be merged and decide how to handle name collisions.
raw = demo.build_raw_dataset()
processed = demo.build_processed_dataset()
print("Raw channels:", sorted(raw.channels))
print("Processed channels:", sorted(processed.channels))

# Merge with collision='rename' so the derived GR channel remains available
# without replacing the raw GR curve.
merged = (
    DatasetBuilder(name="merged")
    .merge(raw, merge_well_metadata=True, merge_provenance=True)
    .merge(
        processed,
        collision="rename",
        rename_template="{mnemonic}_{dataset}",
    )
    .build()
)

print("Merged channels:", sorted(merged.channels))
print("Merge history:", merged.provenance["merge_history"])
print("Renamed channel metadata:", merged.get_channel("GR_qc").metadata)


## Adapt Dataset Merge Walkthrough Safely

- Use `collision='rename'` when both the raw and processed versions of a mnemonic need to stay visible.
- Use provenance merges when downstream users need to understand where each channel came from.
- Promote the merged dataset to your layout layer only after the final channel names are stable.
